In [ ]:
# Configure GOOGLE_API_KEY in your environment.

In [ ]:
# -----------------------------------------------------------------
# 最终版 - 专为 JUPYTER / VS Code .ipynb
# (高性能 - 多线程版本)
#
# API: 已替换为 SiliconFlow (api.siliconflow.cn)
# -----------------------------------------------------------------

import cv2
import time
import requests # ### <--- 1. 导入 requests
from deepface import DeepFace
from pythonosc import udp_client
import NDIlib as ndi
import numpy as np
import os
import importlib.metadata
import threading

# --- 0. 诊断 ---
try:
    print(f"--- [系统] Python 路径: {os.sys.executable}")
    print(f"--- [系统] Python 版本: {importlib.metadata.sys.version.split(' ')[0]}")
    # (移除了 Gemini 诊断)
    print(f"--- [系统] NDI 库版本: {importlib.metadata.version('NDI-python')}")
    print(f"--- [系统] Requests 库版本: {importlib.metadata.version('requests')}")
except Exception:
    pass 

print("\n--- [系统] 脚本开始运行 ---")

# --- 1. 初始化 (准备代码) ---

# ### <--- 2. SiliconFlow API 设置
# Read SILICONFLOW_API_KEY from the environment.
SILICONFLOW_API_KEY = os.environ.get("SILICONFLOW_API_KEY", "").strip() 
SILICONFLOW_URL = "https://api.siliconflow.cn/v1/chat/completions"
SILICONFLOW_HEADERS = {
    "Authorization": f"Bearer {SILICONFLOW_API_KEY}",
    "Content-Type": "application/json"
}
if not SILICONFLOW_API_KEY:
    print("🔴 [错误] 请设置 SILICONFLOW_API_KEY 环境变量")
    raise RuntimeError("SILICONFLOW_API_KEY 环境变量未设置")
else:
    print("✅ [初始化] SiliconFlow API 凭据已加载")

# ### <--- (Google Gemini API 初始化块已被完全移除)

# OSC: 发送数据到TD
try:
    TD_IP = "127.0.0.1"
    TD_PORT = 9000
    osc_client = udp_client.SimpleUDPClient(TD_IP, TD_PORT)
    print("✅ [初始化] OSC 客户端准备就绪 (发送到 127.0.0.1:9000)")
except Exception as e:
    print(f"🔴 [错误] OSC 客户端初始化失败: {e}")
    raise e

# NDI: (保持不变)
ndi_send = None
video_frame = None
try:
    print("... [初始化] 正在启动 NDI ...")
    if not ndi.initialize():
        raise Exception("无法初始化 NDI")
    send_settings = ndi.SendCreate()
    send_settings.ndi_name = "PythonWebcam"
    ndi_send = ndi.send_create(send_settings)
    video_frame = ndi.VideoFrameV2()
    print(f"✅ [初始化] NDI 发送器已创建 (源名称: 'PythonWebcam')")
except Exception as e:
    print(f"🔴 [错误] NDI 初始化失败: {e}")
    raise e

# Webcam: (保持不变)
cap = None
try:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise Exception("🔴 无法打开 Webcam (ID: 0)。请检查macOS摄像头权限！")
    print("✅ [初始化] Webcam 启动成功")
except Exception as e:
    print(f"🔴 [错误] Webcam 打开失败: {e}")
    raise e

# --- 2. 多线程设置 (DeepFace 预热) ---
print("... [初始化] 正在准备 AI 分析 (DeepFace)...")
try:
    _ = DeepFace.analyze(np.zeros((100, 100, 3)), actions=['age'], enforce_detection=False)
    print("✅ [初始化] DeepFace AI 模型已预热 (或已下载)")
except Exception as e:
    print(f"🔴 [错误] DeepFace 预热失败: {e}")

latest_frame = None
frame_lock = threading.Lock() 
analysis_interval = 5 
last_gender = "person"
last_age = 30
ai_thread_running = True

# --- 3. AI 工作线程 (后台循环) ---

def ai_worker_thread():
    """
    (已更新)
    这个线程现在调用 SiliconFlow API。
    """
    global latest_frame, last_gender, last_age
    print(f"--- ✅ [AI 线程] AI 工作线程已启动，每 {analysis_interval} 秒分析一次 ---")
    
    while ai_thread_running:
        try:
            time.sleep(analysis_interval)
            
            current_frame = None
            with frame_lock:
                if latest_frame is not None:
                    current_frame = latest_frame.copy() 

            if current_frame is None: continue

            print("\n--- ⏰ [AI 线程] (1/4) 开始执行图像分析 ---")
            print("... [AI 线程] (2/4) 正在调用 DeepFace (性别/年龄)...")
            analysis = DeepFace.analyze(
                current_frame, 
                actions=['age', 'gender'], 
                enforce_detection=False
            )
            print(f"--- [AI 线程] DeepFace 原始(Raw)分析结果: {analysis} ---")
            
            if isinstance(analysis, list) and len(analysis) > 0:
                result = analysis[0]
                last_gender = result['dominant_gender']
                last_age = result['age']
                print(f"✅ [AI 线程] (3/4) DeepFace: {last_gender}, {last_age} 岁")
            else:
                print("⚠️ [AI 线程] (3/4) DeepFace 未检测到清晰面部")

            osc_client.send_message("/age", last_age)
            osc_client.send_message("/gender", last_gender)

            # ### <--- 4. (已替换) 调用 SiliconFlow API
            
            print("... [AI 线程] (4/4) 正在调用 SiliconFlow API (10秒超时)...")
            
            # 准备 Prompt
            # (注意: DeepFace 返回 'Man' 或 'Woman')
            gender_prompt = "Man" if last_gender == "Man" else "Woman" 
            
            prompt_content = f"""
# 角色 (Role)
你是一个专业的“个人简介”生成AI。你的任务是根据用户的[性别]和[年龄]为他们创造一个“最有吸引力”的社交媒体人设。

# 任务 (Task)
根据我提供的[性别]和[年龄]变量，生成一段第一人称的“社交媒体简介”，必须包含“标题”和“简介”两部分。

# 变量 (Variables)
* 性别: {gender_prompt}
* 年龄: {last_age}

# 核心指令：人设剧本 (Persona Scripts)
你的生成**必须**严格从以下两个剧本中选择一个，不允许混合或创造。

---
## 剧本 1: IF [性别] = 'Man'
* **人设原型 (Persona):** “阿尔法挑战者” (The Alpha Challenger)
* **必须生成的标题 (Title):** 理性主义 | 挑战者 | 极简
* **简介必须包含的核心概念 (Concepts):**
    1.  身份：工作狂 (Workaholic)
    2.  世界观：用“逻辑”和“数据”看待世界。
    3.  痴迷点 (Obsessions)：效率 (efficiency), 科技 (tech), 竞技体育 (competitive sports)。
    4.  活动 (Activities)：健身房 (gym), 研究新工具 (researching tools)。
    5.  社交需求：高效 (efficient), 坦诚 (direct), 有目标 (goal-oriented)。

## 剧本 2: IF [性别] = 'Woman'
* **人设原型 (Persona):** “感性探索家” (The Sensory Explorer)
* **必须生成的标题 (Title):** 感性生活家 | 探索者 | 联结
* **简介必须包含的核心概念 (Concepts):**
    1.  身份：相信直觉 (Intuitive)
    2.  世界观：用“情感”和“体验”感知世界。
    3.  热爱点 (Passions)：艺术 (art), 旅行 (travel), 咖啡 (coffee), 深度对话 (deep talk)。
    4.  活动 (Activities)：书店 (bookstore), 与朋友相聚 (social gathering)。
    5.  社交需求：真实 (authentic), 有意义 (meaningful), 温暖 (warm)。

---

# 执行 (Execute)
请立即为 [性别: {gender_prompt}, 年龄: {last_age}] 的用户生成简介。
"""
            
            # 准备 Payload
            payload = {
                "model": "Qwen/Qwen2.5-7B-Instruct", # <-- 我换成了通义千问7B，它又快又好
                "messages": [
                    { "role": "user", "content": prompt_content }
                ]
            }

            gemini_text = "..." # 默认值
            try:
                # 带有“超时”的 API 调用
                response = requests.post(SILICONFLOW_URL, 
                                         json=payload, 
                                         headers=SILICONFLOW_HEADERS, 
                                         timeout=10) # 10秒超时
                
                # 检查 HTTP 状态码
                if response.status_code == 200:
                    # 成功，解析 JSON
                    json_response = response.json()
                    gemini_text = json_response['choices'][0]['message']['content'].replace("\n", " ")
                    print(f"✅ [AI 线程] SiliconFlow: {gemini_text}")
                else:
                    # API 返回了错误 (例如 401, 403, 500)
                    print(f"🔴 [错误] SiliconFlow API 返回状态码: {response.status_code}")
                    print(f"   错误详情: {response.text}")
                    gemini_text = f"API Error: {response.status_code}"

            except requests.exceptions.Timeout:
                print("🔴 [错误] SiliconFlow API 超时！(10秒)。")
            except requests.exceptions.ConnectionError:
                print("🔴 [错误] SiliconFlow API 连接失败。")
            except Exception as e:
                # 捕捉其他可能的 API 错误 (比如 JSON 解析失败)
                print(f"🔴 [错误] SiliconFlow API 调用失败: {e}")
            
            # ### --- API 调用结束 ---
            
            osc_client.send_message("/gemini_text", gemini_text) 
            print(f"--- ✅ [AI 线程] 分析完毕，OSC 已发送。---")

        except Exception as e:
            if not ai_thread_running: break 
            print(f"🔴 [错误] 在 AI 线程中发生严重错误: {e}")
            time.sleep(analysis_interval) 

# --- 4. 启动 AI 线程 ---
threading.Thread(target=ai_worker_thread, daemon=True).start()


# --- 5. 主循环 (高速 NDI 视频流) ---
print(f"\n--- 🚀 Python中枢已全部启动 ---")
print("\n" + "="*40)
print("   ✅ (主线程) NDI 视频循环已开始运行。")
print("   ✅ (AI 线程) 已在后台启动。")
print("   👀 视频流在 TouchDesigner (NDI In) 中可见。")
print("   ⏹️ 要停止此脚本，请按此单元格的 'Interrupt' (⏹) 按钮")
print("="*40 + "\n")

try:
    while True:
        # (主循环保持不变)
        ret, frame = cap.read()
        if not ret:
            print("🔴 [主线程] Webcam 读取帧失败...")
            time.sleep(0.1) 
            continue
        
        with frame_lock:
            latest_frame = frame 
        
        frame_rgba = cv2.cvtColor(frame, cv2.COLOR_BGR2RGBA)
        
        video_frame.data = frame_rgba
        video_frame.FourCC = ndi.FOURCC_VIDEO_TYPE_RGBA 
        
        ndi.send_send_video_v2(ndi_send, video_frame)
        
except KeyboardInterrupt:
    print("\n--- [系统] 检测到内核中断 (Kernel Interrupt) ---")

finally:
    # --- 6. 清理 (主线程) ---
    print("\n--- [系统] 正在关闭所有服务... ---")
    ai_thread_running = False 
    
    if cap and cap.isOpened():
        cap.release()
        print("✅ [系统] Webcam 已释放")
    
    if ndi_send:
        ndi.send_destroy(ndi_send)
        print("✅ [系统] NDI 发送器已销毁")
    
    ndi.destroy()
    print("✅ [系统] NDI 已释放。程序安全退出。")

In [ ]:
import NDIlib as ndi
print(dir(ndi))
https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5

In [ ]:
import urllib.request
import ssl

try:
    # 尝试访问一个标准的、受信任的 HTTPS 网站
    context = ssl.create_default_context()
    with urllib.request.urlopen('https://www.google.com', context=context, timeout=5) as response:
        print("✅ [诊断] HTTPS 连接成功！")
        print(f"   状态码 (Status Code): {response.status}")
        print("   网络和 SSL 证书看起来是正常的。")

except Exception as e:
    print("🔴 [诊断] HTTPS 连接失败！")
    print(f"   错误: {e}")
    print("\n   --- 这很可能就是您 Gemini 超时的根本原因 ---")
    if "CERTIFICATE_VERIFY_FAILED" in str(e):
        print("   --- 诊断结果: 您的 Python 环境 SSL 证书已损坏或丢失。 ---")